In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import os
import re
import matplotlib.pyplot as plt


In [2]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#determines who is ahead at a specific event (row) in the game
def ahead_team(n_row,goals_all):
    away_score=0
    home_score=0
    for idx,row in goals_all[goals_all['n_row']<n_row].iterrows():
        if (row['description'].split(' ')[0]==row['away_team']):
            away_score+=1
        else:
            home_score+=1
    if (away_score>home_score):
        return('away')
    elif (away_score<home_score):
        return('home')
    else:
        return('none')

In [55]:
def get_goalie_pulls(df):
    
    #convert to total seconds to end of the game
    df['total_sec']=df['minute'].astype(int)*60+df['second'].astype(int)
    #deal with the format
    df['n_row']=df['n_row'].apply(lambda x: int(x))
    
    #replace empty line cells with a G so they will not be mistaken for empty nets
    df.loc[df['away_line'].isna(),'away_line']='G'
    df.loc[df['home_line'].isna(),'home_line']='G'
    
    #save the game goals and who scored them
    goals_all=df[df.event=='GOAL']
    
    #determining the events without a goalie by checking individual lines without G in team lines during the third period
    goalie_out=df.loc[((~df['away_line'].str.contains('G'))|(~df['home_line'].str.contains('G')))&
                     (df['total_sec']<600)&(df['period']==3)]
    #check who is winning during the goalie out
    goalie_out['winning']=goalie_out.n_row.apply(lambda x: ahead_team(x,goals_all))
    
    if len(goalie_out):
        #delayed penalty
        delayed_penalty=goalie_out[((goalie_out['home_line'].str.contains('G'))&(goalie_out.winning=='away'))|
                                   ((goalie_out['away_line'].str.contains('G'))&(goalie_out.winning=='home'))]
        if len(delayed_penalty):
            goalie_out=goalie_out.drop(delayed_penalty.index)
            if (len(goalie_out)==0):
                return None
            
        # Ignore pulls where any penalty occurs during the empty net window
        if goalie_out['event'].str.contains('PENL').any():
            return None
        
        # Check the row immediately after goalie_out ends
        last_goalie_out_row = goalie_out['n_row'].max()
        next_row = df[df['n_row'] == last_goalie_out_row + 1]

        if len(next_row) and next_row['event'].values[0] == 'PENL':
            if ('G' in next_row['away_line'].values[0] and 'G' in next_row['home_line'].values[0]):
        # Delayed penalty false positive — goalie came back at the whistle
                return None
        
        goalie_out_sec=goalie_out.total_sec.values[0]
        

        success=0
        goal=goalie_out.loc[(goalie_out.event=='GOAL')]    
        if len(goal):
            if(goal['away_line'].str.contains('G').values[0] and 
               (goal.description.values[0].split(' ')[0]==goal.away_team.values[0])):
                success=-1
            elif(goal['home_line'].str.contains('G').values[0] and 
                 (goal.description.values[0].split(' ')[0]==goal.home_team.values[0])):
                success=-1
            else:
                success=1
                
        return([goalie_out_sec,success])

In [56]:
%%time
pull_time=[]
success=[]
game=[]
goal_number=[]

for file in os.listdir('./data/game/'):    
    game_data = pd.read_csv('./data/game/' + file)
    
    result = get_goalie_pulls(game_data)
    if result is not None:
        game_pull_time, game_success = result
        pull_time.append(game_pull_time)
        success.append(game_success)
        game.append(file)
    
    goal_number.append(len(game_data[(game_data['event'] == 'GOAL')
                                     & (game_data['period'] == 3)
                                     & (game_data['minute'] > 5)]))


CPU times: total: 1min 3s
Wall time: 1min 4s


In [54]:
%%time
# Diagnostic: count games with multiple goalie pull events
multiple_pulls = 0
total_pulls = 0

for file in os.listdir('./data/game/'):
    game_data = pd.read_csv('./data/game/' + file)
    game_data['total_sec'] = game_data['minute'].astype(int)*60 + game_data['second'].astype(int)
    game_data['n_row'] = game_data['n_row'].astype(int)
    game_data.loc[game_data['away_line'].isna(), 'away_line'] = 'G'
    game_data.loc[game_data['home_line'].isna(), 'home_line'] = 'G'
    
    goalie_out = game_data.loc[
        (~game_data['away_line'].str.contains('G') | ~game_data['home_line'].str.contains('G')) &
        (game_data['total_sec'] < 560) & (game_data['period'] == 3)]

    
    
    # Skip if any penalty occurred during the empty net window
    if goalie_out['event'].str.contains('PENL').any():
        continue


    # Check the row immediately after goalie_out ends
    last_goalie_out_row = goalie_out['n_row'].max()
    next_row = game_data[game_data['n_row'] == last_goalie_out_row + 1]

    if len(next_row) and next_row['event'].values[0] == 'PENL':
        if ('G' in next_row['away_line'].values[0] and 
            'G' in next_row['home_line'].values[0]):
        # Delayed penalty false positive — goalie came back at the whistle
            continue


    if len(goalie_out) > 1:
        gaps = goalie_out['n_row'].diff() > 10
        if gaps.any():
            multiple_pulls += 1
    
    if len(goalie_out) > 0:
        total_pulls += 1
       # if (max(goalie_out['total_sec'])>400):
        #    print(max(goalie_out['total_sec']))
         #   print(a)

print(f'Total games with a goalie pull: {total_pulls}')
print(f'Games with multiple pull events: {multiple_pulls}')
print(f'Percentage: {round(100 * multiple_pulls / total_pulls, 1)}%')

Total games with a goalie pull: 5002
Games with multiple pull events: 39
Percentage: 0.8%
CPU times: total: 47 s
Wall time: 3min 20s


In [57]:
#save the processed data
pd.DataFrame(list(zip(game, pull_time,success)),
                 columns =['game', 'total_sec','success']).to_csv('./data/processed/goalie_pull_data_2011_2021.csv',index=False)


In [28]:
print('Probablity of normal goal per second for two teams was',
      round(np.sum(goal_number)/(14*60)/len(os.listdir('./data/game/')),5))

Probablity of normal goal per second for two teams was 0.00141
